In [1]:
"""
BDD100K → YOLO Conversion Script for Tamakkan
==============================================
What this does:
- Reads per-image JSON label files from BDD100K
- Removes the 'train' class entirely
- Merges bike + rider + motor → vulnerable_road_user
- Converts bounding boxes to YOLO normalized format
- Writes one .txt label file per image
- Generates data.yaml

New class mapping:
    0: car
    1: truck
    2: bus
    3: person
    4: traffic light
    5: traffic sign
    6: vulnerable_road_user  (merged: bike + rider + motor)
"""

import json
import os
from pathlib import Path

In [2]:

# ── CONFIGURE THESE PATHS ──────────────────────────────────────────────────────

IMAGES_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\bdd100k_images_100k\100k"
LABELS_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\bdd100k_labels\100k"

# Where the converted YOLO dataset will be saved
OUTPUT_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2"

In [3]:

# ── CLASS MAPPING ──────────────────────────────────────────────────────────────
 
# Classes to completely ignore (drop these annotations)
IGNORE_CLASSES = {"train"}
 
# Classes to merge into vulnerable_road_user
MERGE_INTO_VRU = {"bike", "rider", "motor"}

# Final class ID mapping
CLASS_MAP = {
    "car":                  0,
    "truck":                1,
    "bus":                  2,
    "person":               3,
    "traffic light":        4,
    "traffic sign":         5,
    "vulnerable_road_user": 6,  # bike + rider + motor merged here
}
 
IMAGE_WIDTH  = 1280  # BDD100K standard image size
IMAGE_HEIGHT = 720
 
SPLITS = ["train", "val", "test"]

In [4]:

# ── CONVERSION LOGIC ───────────────────────────────────────────────────────────
 
def convert_bbox_to_yolo(x1, y1, x2, y2, img_w, img_h):
    """Convert BDD100K (x1,y1,x2,y2) absolute bbox to YOLO (cx,cy,w,h) normalized."""
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    # Clamp to [0, 1] to handle any edge annotations
    cx = max(0.0, min(1.0, cx))
    cy = max(0.0, min(1.0, cy))
    w  = max(0.0, min(1.0, w))
    h  = max(0.0, min(1.0, h))
    return cx, cy, w, h
 
 
def process_split(split: str, stats: dict):
    """Process one split (train/val/test) and write YOLO labels."""
    labels_dir = Path(LABELS_ROOT) / split
    output_labels_dir = Path(OUTPUT_ROOT) / "labels" / split
    output_labels_dir.mkdir(parents=True, exist_ok=True)
 
    json_files = list(labels_dir.glob("*.json"))
    print(f"\n[{split}] Found {len(json_files)} JSON label files")
 
    for json_path in json_files:
        with open(json_path, "r") as f:
            data = json.load(f)
 
        # BDD100K per-image JSON structure:
        # { "frames": [{ "objects": [{ "category": ..., "box2d": {...} }] }] }
        # Some files may use a flat structure — we handle both
        objects = []
        if "frames" in data:
            for frame in data["frames"]:
                objects.extend(frame.get("objects", []))
        elif "objects" in data:
            objects = data["objects"]
 
        yolo_lines = []
 
        for obj in objects:
            category = obj.get("category", "").strip().lower()
            box2d    = obj.get("box2d", None)
 
            # Skip if no bounding box
            if box2d is None:
                stats["no_box"] += 1
                continue
 
            # Skip ignored classes
            if category in IGNORE_CLASSES:
                stats["ignored"] += 1
                continue
 
            # Merge VRU classes
            if category in MERGE_INTO_VRU:
                class_id = CLASS_MAP["vulnerable_road_user"]
                stats["vru_merged"] += 1
            elif category in CLASS_MAP:
                class_id = CLASS_MAP[category]
            else:
                # Unknown class — skip and log
                stats["unknown"].add(category)
                stats["unknown_count"] += 1
                continue
 
            x1 = box2d["x1"]
            y1 = box2d["y1"]
            x2 = box2d["x2"]
            y2 = box2d["y2"]
 
            # Skip degenerate boxes
            if x2 <= x1 or y2 <= y1:
                stats["degenerate"] += 1
                continue
 
            cx, cy, w, h = convert_bbox_to_yolo(x1, y1, x2, y2, IMAGE_WIDTH, IMAGE_HEIGHT)
            yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            stats["total_annotations"] += 1
            stats["per_class"][class_id] += 1
 
        # Write .txt file (even if empty — means image has no relevant objects)
        txt_filename = json_path.stem + ".txt"
        with open(output_labels_dir / txt_filename, "w") as f:
            f.write("\n".join(yolo_lines))
 
        stats["images_processed"] += 1
 
    print(f"[{split}] Done. Written to: {output_labels_dir}")
 
 
def write_data_yaml():
    """Write the data.yaml file for YOLO training."""
    yaml_content = f"""# Tamakkan - YOLOv11s Dataset Config
# Classes: 7 (merged bike+rider+motor → vulnerable_road_user)
 
path: {OUTPUT_ROOT}
train: images/train
val:   images/val
test:  images/test
 
nc: 7
names:
  0: car
  1: truck
  2: bus
  3: person
  4: traffic light
  5: traffic sign
  6: vulnerable_road_user
"""
    yaml_path = Path(OUTPUT_ROOT) / "data.yaml"
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    with open(yaml_path, "w") as f:
        f.write(yaml_content)
    print(f"\ndata.yaml written to: {yaml_path}")
 
 
def print_summary(stats: dict):
    class_names = {
        0: "car", 1: "truck", 2: "bus", 3: "person",
        4: "traffic light", 5: "traffic sign", 6: "vulnerable_road_user"
    }
    print("\n" + "="*55)
    print("CONVERSION SUMMARY")
    print("="*55)
    print(f"  Images processed      : {stats['images_processed']:,}")
    print(f"  Total annotations     : {stats['total_annotations']:,}")
    print(f"  VRU merged (b+r+m)    : {stats['vru_merged']:,}")
    print(f"  Ignored (train class) : {stats['ignored']:,}")
    print(f"  Skipped (no box)      : {stats['no_box']:,}")
    print(f"  Skipped (degenerate)  : {stats['degenerate']:,}")
    print(f"  Unknown classes       : {stats['unknown_count']:,} → {stats['unknown']}")
    print("\n  Per-class annotation counts:")
    for cid, count in sorted(stats["per_class"].items()):
        print(f"    [{cid}] {class_names[cid]:<22}: {count:,}")
    print("="*55)


In [5]:

# ── MAIN ───────────────────────────────────────────────────────────────────────
 
if __name__ == "__main__":
    stats = {
        "images_processed":   0,
        "total_annotations":  0,
        "vru_merged":         0,
        "ignored":            0,
        "no_box":             0,
        "degenerate":         0,
        "unknown_count":      0,
        "unknown":            set(),
        "per_class":          {i: 0 for i in range(7)},
    }
 
    print("Tamakkan BDD100K → YOLO Conversion")
    print(f"Output: {OUTPUT_ROOT}\n")
 
    # NOTE: Images don't need to be moved — just symlink or copy if needed.
    # YOLO can read images from a different path than labels as long as
    # data.yaml points correctly. But for cleanliness, you can also copy images.
    # This script only converts labels. Images stay where they are.
    # Update data.yaml image paths if you keep images in original location.
 
    for split in SPLITS:
        process_split(split, stats)
 
    write_data_yaml()
    print_summary(stats)
 
    print("\nNext step: update data.yaml image paths if needed, then run training.")
    print("Images are NOT copied — update path in data.yaml to point to your existing images folder.")

Tamakkan BDD100K → YOLO Conversion
Output: C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2


[train] Found 70000 JSON label files
[train] Done. Written to: C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2\labels\train

[val] Found 10000 JSON label files
[val] Done. Written to: C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2\labels\val

[test] Found 20000 JSON label files
[test] Done. Written to: C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2\labels\test


UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 76: character maps to <undefined>

In [6]:
from pathlib import Path

OUTPUT_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2"

yaml_content = """# Tamakkan - YOLOv11s Dataset Config
# Classes: 7 (merged bike+rider+motor into vulnerable_road_user)

path: C:/Users/Admin/Desktop/Grad Project/Code/Datasets/BDD100k_V2
train: images/train
val:   images/val
test:  images/test

nc: 7
names:
  0: car
  1: truck
  2: bus
  3: person
  4: traffic light
  5: traffic sign
  6: vulnerable_road_user
"""

yaml_path = Path(OUTPUT_ROOT) / "data.yaml"
yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"data.yaml written to: {yaml_path}")

data.yaml written to: C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2\data.yaml


In [7]:
print("="*55)
print("CONVERSION SUMMARY")
print("="*55)
print(f"  Images processed      : {stats['images_processed']:,}")
print(f"  Total annotations     : {stats['total_annotations']:,}")
print(f"  VRU merged (b+r+m)    : {stats['vru_merged']:,}")
print(f"  Ignored (train class) : {stats['ignored']:,}")
print(f"  Unknown classes       : {stats['unknown_count']:,} -> {stats['unknown']}")

class_names = {0:"car",1:"truck",2:"bus",3:"person",4:"traffic light",5:"traffic sign",6:"vulnerable_road_user"}
print("\n  Per-class annotation counts:")
for cid, count in sorted(stats["per_class"].items()):
    print(f"    [{cid}] {class_names[cid]:<22}: {count:,}")

CONVERSION SUMMARY
  Images processed      : 100,000
  Total annotations     : 1,840,978
  VRU merged (b+r+m)    : 20,992
  Ignored (train class) : 179
  Unknown classes       : 0 -> set()

  Per-class annotation counts:
    [0] car                   : 1,021,517
    [1] truck                 : 42,949
    [2] bus                   : 16,498
    [3] person                : 129,308
    [4] traffic light         : 265,926
    [5] traffic sign          : 343,793
    [6] vulnerable_road_user  : 20,987


In [8]:
from pathlib import Path

OUTPUT_ROOT = r"C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2"

yaml_content = """# Tamakkan - YOLOv11s Dataset Config
# Classes: 7 (merged bike+rider+motor into vulnerable_road_user)

path: C:/Users/Admin/Desktop/Grad Project/Code/Datasets/BDD100k_V2
train: C:/Users/Admin/Desktop/Grad Project/Code/Datasets/bdd100k_images_100k/100k/train
val:   C:/Users/Admin/Desktop/Grad Project/Code/Datasets/bdd100k_images_100k/100k/val
test:  C:/Users/Admin/Desktop/Grad Project/Code/Datasets/bdd100k_images_100k/100k/test

nc: 7
names:
  0: car
  1: truck
  2: bus
  3: person
  4: traffic light
  5: traffic sign
  6: vulnerable_road_user
"""

yaml_path = Path(OUTPUT_ROOT) / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"data.yaml updated at: {yaml_path}")

data.yaml updated at: C:\Users\Admin\Desktop\Grad Project\Code\Datasets\BDD100k_V2\data.yaml
